In [0]:
%sh
java -jar /Volumes/ing_hackathon/jars/parsers/japicmp.jar \
  --old /Volumes/ing_hackathon/jars/libraries/jackson-databind-2.5.2.jar \
  --new /Volumes/ing_hackathon/jars/libraries/jackson-databind-2.21.0.jar \
  --only-modified \
  --only-incompatible \
  --ignore-missing-classes \
  --include-synthetic


In [0]:
%sh
java -jar /Volumes/ing_hackathon/jars/parsers/japicmp.jar \
  --old /Volumes/ing_hackathon/jars/libraries/jackson-databind-2.5.2.jar \
  --new /Volumes/ing_hackathon/jars/libraries/jackson-databind-2.21.0.jar \
  --only-modified \
  --only-incompatible \
  --ignore-missing-classes \
  --include-synthetic \
  > /Volumes/ing_hackathon/jars/diff/japicmp_jackson-databind_2.5.2_to_2.21.0.txt


In [0]:
%sh
cat /Volumes/ing_hackathon/jars/diff/japicmp_jackson-databind_2.5.2_to_2.21.0.txt



In [0]:
import subprocess, textwrap, re
import pandas as pd

cmd = [
    "java", "-jar", "/Volumes/ing_hackathon/jars/parsers/japicmp.jar",
    "--old", "/Volumes/ing_hackathon/jars/libraries/jackson-databind-2.5.2.jar",
    "--new", "/Volumes/ing_hackathon/jars/libraries/jackson-databind-2.21.0.jar",
    "--only-modified",
    "--only-incompatible",
    "--ignore-missing-classes",
    "--include-synthetic",
]

p = subprocess.run(cmd, capture_output=True, text=True)
report = (p.stdout or "") + ("\n" + p.stderr if p.stderr else "")

# --- parse into rows ---
header_re = re.compile(r'^\*{3}([!+=-]{1,2})\s+(MODIFIED|REMOVED|NEW|UNCHANGED)\s+(CLASS|INTERFACE|ENUM|ANNOTATION|MODULE):\s+(.*)$')
detail_re = re.compile(r'^\s*([+\-=]{3}[!+=-])\s+(REMOVED|NEW|UNCHANGED|MODIFIED)\s+(METHOD|FIELD|CONSTRUCTOR|SUPERCLASS|INTERFACE|EXCEPTION|CLASS FILE FORMAT VERSION|TYPE):\s*(.*)$')

rows = []
current = None

for line in report.splitlines():
    m = header_re.match(line)
    if m:
        current = {
            "change": m.group(2),
            "kind": m.group(3),
            "symbol": m.group(1),
            "target": m.group(4).strip(),
        }
        continue

    d = detail_re.match(line)
    if d and current:
        rows.append({
            "change": current["change"],
            "kind": current["kind"],
            "symbol": current["symbol"],
            "target": current["target"],
            "detail_change": d.group(2),
            "detail_kind": d.group(3),
            "detail": d.group(4).strip(),
            "rawline": line,
        })

df = pd.DataFrame(rows)
df["name"] = df["target"].str.extract(r'(com\.[\w\.\$]+)')[0].fillna(df["target"])

display(df)


In [0]:
import subprocess

cmd = [
    "java", "-jar", "/Volumes/ing_hackathon/jars/parsers/japicmp.jar",
    "--old", "/Volumes/ing_hackathon/jars/libraries/jackson-databind-2.5.2.jar",
    "--new", "/Volumes/ing_hackathon/jars/libraries/jackson-databind-2.21.0.jar",
    "--only-modified",
    "--only-incompatible",
    "--ignore-missing-classes",
    "--include-synthetic",
]

p = subprocess.run(cmd, capture_output=True, text=True)
report = (p.stdout or "") + ("\n" + p.stderr if p.stderr else "")

# Optional sanity check (small preview)
report.splitlines()[:5]


In [0]:
import re
import pandas as pd

# Header like:
# ***! MODIFIED CLASS: PUBLIC ABSTRACT com.fasterxml.... (notes)
header_re = re.compile(
    r'^\*{3}([!+=-]{1,2})\s+'
    r'(MODIFIED|REMOVED|NEW|UNCHANGED)\s+'
    r'(CLASS|INTERFACE|ENUM|ANNOTATION|MODULE):\s+'
    r'(.+?)\s*$'
)

# Detail like:
# ---! REMOVED METHOD: PUBLIC(-) returnType name(params...)
detail_re = re.compile(
    r'^\s*([+\-=]{3}[!+=-])\s+'
    r'(REMOVED|NEW|UNCHANGED|MODIFIED)\s+'
    r'(METHOD|FIELD|CONSTRUCTOR|SUPERCLASS|INTERFACE|EXCEPTION|CLASS FILE FORMAT VERSION|TYPE):\s*'
    r'(.+?)\s*$'
)

# Extract FQCN from header tail
fqcn_re = re.compile(r'(com\.[\w\.\$]+)')

# Extract method signature pieces from the detail text
# e.g. "PUBLIC(-) java.lang.Class<?> findX(a.b.C, d.E)"
method_sig_re = re.compile(
    r'^(?P<mods>.+?)\s+'
    r'(?P<ret>[\w\.\$\?<>\[\]]+)\s+'
    r'(?P<name>[\w\$]+)\((?P<params>.*)\)\s*$'
)

rows = []
current = None

for line in report.splitlines():
    m = header_re.match(line)
    if m:
        target = m.group(4)
        fqcn = fqcn_re.search(target)
        current = {
            "class_change": m.group(2),           # MODIFIED/REMOVED/NEW...
            "class_kind": m.group(3),             # CLASS/INTERFACE...
            "class_symbol": m.group(1),           # ***! etc
            "class_header": target,
            "fqcn": fqcn.group(1) if fqcn else None,
            "class_notes": target.replace(fqcn.group(1), "").strip() if fqcn else target,
        }
        continue

    d = detail_re.match(line)
    if d and current:
        detail_text = d.group(4)

        method_name = None
        ret = None
        params = None

        if d.group(3) in ("METHOD", "CONSTRUCTOR"):
            ms = method_sig_re.match(detail_text)
            if ms:
                method_name = ms.group("name")
                ret = ms.group("ret")
                params = ms.group("params")

        rows.append({
            "class_name": current["fqcn"],
            "class_kind": current["class_kind"],
            "class_change": current["class_change"],
            "class_notes": current["class_notes"],

            "member_change": d.group(2),          # REMOVED/NEW/MODIFIED...
            "member_kind": d.group(3),            # METHOD/FIELD/...
            "member_text": detail_text,

            "member_name": method_name,
            "return_type": ret,
            "param_types": params,
            "rawline": line,
        })

DiffTable = pd.DataFrame(rows)

# A simple “break risk” score
def break_risk(row):
    if row["member_change"] == "REMOVED" and row["member_kind"] in ("METHOD", "CONSTRUCTOR", "FIELD"):
        return "HIGH"
    if row["member_change"] == "MODIFIED" and row["member_kind"] in ("METHOD", "CONSTRUCTOR", "FIELD", "TYPE"):
        return "MEDIUM"
    return "LOW"

DiffTable["break_risk"] = DiffTable.apply(break_risk, axis=1)

# Helpful “what to search for” patterns (for Java/Scala)
DiffTable["search_hint"] = DiffTable.apply(
    lambda r: (f"{r['class_name']}.{r['member_name']}" if pd.notna(r["member_name"]) and pd.notna(r["class_name"]) else None),
    axis=1
)

# Column you can fill later with actual migration guidance
DiffTable["migration_notes"] = ""

display(DiffTable)


In [0]:
%sh
java -jar /Workspace/Users/labuser13683806_1770622817@vocareum.com/java-lib-usage/target/java-lib-usage-1.0.0.jar /Workspace/Users/labuser13683806_1770622817@vocareum.com/java-lib-usage/src/main/java/com/example/libusage/App.java




In [0]:
%sh

curl -L -o /Volumes/ing_hackathon/jars/libraries/jackson-databind-2.5.2.jar \
https://repo1.maven.org/maven2/com/fasterxml/jackson/core/jackson-databind/2.5.2/jackson-databind-2.5.2.jar

curl -L -o /Volumes/ing_hackathon/jars/libraries/jackson-core-2.5.2.jar \
https://repo1.maven.org/maven2/com/fasterxml/jackson/core/jackson-core/2.5.2/jackson-core-2.5.2.jar

curl -L -o /Volumes/ing_hackathon/jars/libraries/jackson-annotations-2.5.2.jar \
https://repo1.maven.org/maven2/com/fasterxml/jackson/core/jackson-annotations/2.5.2/jackson-annotations-2.5.2.jar


In [0]:
# Databricks notebook cell (Python)
# Runs your java-lib-usage jar against a source file + source dir + jackson classpath from a UC Volume,
# captures stdout, then applies the equivalent of `| sort | uniq`.

import subprocess
import re

# --- Paths (edit only if your locations differ) ---
ANALYZER_JAR = "/Workspace/Users/labuser13683806_1770622817@vocareum.com/java-lib-usage/target/java-lib-usage-1.0.0.jar"
SOURCE_FILE  = "/Volumes/ing_hackathon/mock_data/source_code/App2.java"
SOURCE_DIR   = "/Volumes/ing_hackathon/mock_data/source_code/"

# Jackson 2.5.2 jars stored in your UC Volume
JACKSON_DATABIND = "/Volumes/ing_hackathon/jars/libraries/jackson-databind-2.5.2.jar"
JACKSON_CORE     = "/Volumes/ing_hackathon/jars/libraries/jackson-core-2.5.2.jar"
JACKSON_ANN      = "/Volumes/ing_hackathon/jars/libraries/jackson-annotations-2.5.2.jar"

# --- Sanity checks (optional but useful) ---
for p in [ANALYZER_JAR, SOURCE_FILE, JACKSON_DATABIND, JACKSON_CORE, JACKSON_ANN]:
    ls = subprocess.run(["bash", "-lc", f"ls -l '{p}'"], capture_output=True, text=True)
    if ls.returncode != 0:
        raise FileNotFoundError(f"Missing or not accessible: {p}\n{ls.stderr}")

# --- Read Main-Class from analyzer jar manifest ---
manifest = subprocess.run(
    ["bash", "-lc", f"unzip -p '{ANALYZER_JAR}' META-INF/MANIFEST.MF"],
    capture_output=True,
    text=True
)
if manifest.returncode != 0:
    raise RuntimeError(f"Could not read MANIFEST.MF:\n{manifest.stderr}")

m = re.search(r"^Main-Class:\s*(.+)\s*$", manifest.stdout, re.MULTILINE)
if not m:
    raise RuntimeError("Main-Class not found in MANIFEST.MF. "
                       "Your analyzer jar may not be executable via Main-Class.")
MAIN_CLASS = m.group(1).strip()

# --- Build classpaths ---
# Classpath passed to YOUR TOOL as an argument (like your local quoted string)
analysis_cp = ":".join([JACKSON_DATABIND, JACKSON_CORE, JACKSON_ANN])

# JVM runtime classpath (analyzer jar + libs so it can load Jackson classes while analyzing)
jvm_cp = f"{ANALYZER_JAR}:{analysis_cp}"

# --- Run analyzer (mirrors your local command shape) ---
cmd = ["java", "-cp", jvm_cp, MAIN_CLASS, SOURCE_FILE, SOURCE_DIR, analysis_cp]

result = subprocess.run(cmd, capture_output=True, text=True)

# If the tool writes to stderr, include it (and show errors clearly)
raw_out = (result.stdout or "").strip()
raw_err = (result.stderr or "").strip()

if result.returncode != 0:
    raise RuntimeError(
        "Analyzer failed.\n"
        f"Command: {' '.join(cmd)}\n\n"
        f"STDOUT:\n{raw_out}\n\n"
        f"STDERR:\n{raw_err}\n"
    )

# --- sort | uniq equivalent ---
lines = []
if raw_out:
    lines.extend(raw_out.splitlines())
# sometimes tools print important stuff to stderr even on success
if raw_err:
    lines.extend(raw_err.splitlines())

output = "\n".join(sorted(set(line for line in lines if line.strip())))

# In Databricks, prefer display over print for big outputs
display(output)


In [0]:
# Turn the analyzer output string into a Spark DF (one row per invocation)

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# `output` is the string you already have (newline-separated)
# output = "com....\ncom....\n..."

lines = [ln.strip() for ln in output.splitlines() if ln.strip()]

SourceMethodsInvocation = spark.createDataFrame([(ln,) for ln in lines], ["invocation"])

# Split into class + method(signature)
SourceMethodsInvocation = (
    SourceMethodsInvocation
    .withColumn("class_name", F.regexp_extract("invocation", r"^(.*)\.[^.]+\(", 1))
    .withColumn("method_name", F.regexp_extract("invocation", r"\.([^.]+)\(", 1))
    .withColumn("params_raw", F.regexp_extract("invocation", r"\((.*)\)$", 1))
    .withColumn(
        "param_count",
        F.when(F.col("params_raw") == "", F.lit(0))
         .otherwise(F.size(F.split(F.col("params_raw"), r"\s*,\s*")))
    )
    .withColumn(
        "params",
        F.when(F.col("params_raw") == "", F.array().cast("array<string>"))
         .otherwise(F.split(F.col("params_raw"), r"\s*,\s*"))
    )
    .drop("params_raw")
)

display(SourceMethodsInvocation)


In [0]:
SourceMethodsInvocation.display()

In [0]:
DiffTable.display()

In [0]:
from pyspark.sql import functions as F

filtered_df1 = df1.join( SourceMethodsInvocation.select("class_name").distinct(), df1.ClassName == SourceMethodsInvocation.class_name, "inner" )

display(filtered_df1)

In [0]:
from pyspark.sql import functions as F

filtered_df1 = DiffTable.join(
    SourceMethodsInvocation.select("class_name").distinct(),
    "class_name" == "class_name",
    "left_semi"
)

display(filtered_df1)


In [0]:
from pyspark.sql import functions as F

filtered_df1 = DiffTable.join(
    SourceMethodsInvocation.select("class_name").distinct(),
    F.col("Class name") == F.col("class_name"),
    "left_semi"
)

display(filtered_df1)


In [0]:
display(SourceMethodsInvocation)

In [0]:
display(DiffTable)

In [0]:
DiffTable = spark.createDataFrame(DiffTable)


In [0]:
filtered_df = DiffTable.join(
    SourceMethodsInvocation.select("class_name").distinct(),
    "class_name",
    "inner"
)

display(filtered_df)


In [0]:
from pyspark.sql import functions as F

filtered_df2 = filtered_df.alias("d").join(
    SourceMethodsInvocation.alias("s"),
    F.col("d.rawline").contains(F.col("s.method_name")),
    "left_semi"
)

display(filtered_df2)


In [0]:
table_name = "ing_hackathon.mock_data.test_diff"   # change schema/table as needed

(filtered_df2
 .write
 .mode("overwrite")          # or "append"
 .format("delta")
 .saveAsTable(table_name)
)